##***5.Feature selection for Churn Prediction***

In [1]:
churn_df = df.copy()

NameError: name 'df' is not defined

In [ ]:
churn_df.info()

In [ ]:
churn_df.shape

In [ ]:
churn_df.isnull().sum()

In [ ]:
#Dropping the target label column
churn_df.drop(['churn'], axis=1, inplace = True)

In [ ]:
#Dropping bucket columns as they have been derived from the original features and may lead to redundancy
churn_df.drop(['high_usage', 'account_length_group', 'service_call_group', 'usage_group'], axis=1, inplace = True)

#Dropping redundant average duration columns
churn_df.drop(['avg_day_call_duration', 'avg_eve_call_duration', 'avg_night_call_duration', 'avg_intl_call_duration'], axis=1, inplace=True)

#Dropping one of the ratio duplicates
churn_df.drop(['charge_per_unit'], axis=1, inplace=True)


In [ ]:
churn_df.shape

In [ ]:
churn_df.isnull().sum()

In [ ]:
churn_df.info()

In [ ]:
# To see the column names that are 'object' type
churn_df.select_dtypes(include=['object']).columns

In [2]:
churn_df['state'].nunique()

NameError: name 'churn_df' is not defined

####**Categorical Encoding for churn prediction**

In [ ]:
categorical_cols = [
    'state',
    'area_code',
    'international_plan',
    'voice_mail_plan',
    'customer_segment'
]

churn_df_encoded = pd.get_dummies(churn_df, columns=categorical_cols, drop_first=True)

In [ ]:
churn_df_encoded.shape

####***Feature Selection using Random Forest***

In [ ]:
X_fs = churn_df_encoded.drop('churn_flag', axis=1)
y_fs = churn_df_encoded['churn_flag']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_fs = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
)

rf_fs.fit(X_fs, y_fs)

In [3]:
feature_importance = pd.DataFrame({
    'feature': X_fs.columns,
    'importance': rf_fs.feature_importances_
}).sort_values(by='importance', ascending=False)

feature_importance.head(20)

NameError: name 'pd' is not defined

In [ ]:
top_n = 20

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importance.head(top_n),
    x='importance',
    y='feature'
)
plt.title('Top 20 Feature Importances - Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

####**Let's perform the correltation/redundancy pruning technique**

In [ ]:
top_20_features = [
    'total_charges',
    'international_plan_yes',
    'total_usage',
    'total_day_minutes',
    'number_customer_service_calls',
    'high_service_issue_flag',
    'service_calls_per_month_proxy',
    'day_usage_share',
    'avg_call_duration',
    'calls_per_minute',
    'intl_mins_per_call',
    'total_intl_minutes',
    'charge_per_minute',
    'intl_usage_share',
    'intl_charge_per_call',
    'high_value_customer_flag',
    'total_eve_minutes',
    'account_length',
    'night_usage_share',
    'number_vmail_messages'
]

corr_matrix = X_fs[top_20_features].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Heatmap - Top 20 Selected Features")
plt.show()

In [ ]:
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5
)
plt.title("Correlation Heatmap - Top 20 Selected Features")
plt.show()

🔴 High-correlation pairs we should act on

From heatmap, the important ones are:

1.
avg_call_duration  ↔  calls_per_minute   = -0.98

2.
intl_mins_per_call ↔ intl_charge_per_call = 1.00

3.
day_usage_share ↔ charge_per_minute = 0.94

4.
total_charges ↔ total_usage = 0.89

5.
total_charges ↔ total_day_minutes = 0.88

6.
day_usage_share ↔ total_day_minutes = 0.86

7.
intl_usage_share ↔ total_intl_minutes = 0.85

8.
number_customer_service_calls ↔ service_calls_per_month_proxy = 0.81

9.
avg_call_duration ↔ total_usage = 0.78

10.
charge_per_minute ↔ total_day_minutes = 0.78

11.
total_charges ↔ high_value_customer_flag = 0.73

Features that we are going to drop

    'calls_per_minute',       # almost mirror of avg_call_duration (-0.98)

    'intl_charge_per_call',   # duplicate of intl_mins_per_call (1.00)

    'charge_per_minute',      # heavily overlaps with day_usage_share / total_day_minutes

    'high_value_customer_flag' # compressed threshold version of value/usage


candidate_features_after_hard_pruning = [
  
    'total_charges',
    'international_plan_yes',
    'total_usage',
    'total_day_minutes',
    'number_customer_service_calls',
    'high_service_issue_flag',
    'service_calls_per_month_proxy',
    'day_usage_share',
    'avg_call_duration',
    'intl_mins_per_call',
    'total_intl_minutes',
    'intl_usage_share',
    'total_eve_minutes',
    'account_length',
    'night_usage_share',
    'number_vmail_messages'
]

Now let's create feature sets for model training

In [ ]:
feature_set_A = [
    'total_charges',
    'international_plan_yes',
    'total_day_minutes',
    'number_customer_service_calls',
    'high_service_issue_flag',
    'avg_call_duration',
    'intl_mins_per_call',
    'day_usage_share',
    'account_length',
    'number_vmail_messages'
]

In [ ]:
feature_set_B = [
    'total_charges',
    'international_plan_yes',
    'total_usage',
    'total_day_minutes',
    'number_customer_service_calls',
    'high_service_issue_flag',
    'service_calls_per_month_proxy',
    'day_usage_share',
    'avg_call_duration',
    'intl_mins_per_call',
    'total_intl_minutes',
    'intl_usage_share',
    'total_eve_minutes',
    'account_length',
    'night_usage_share',
    'number_vmail_messages'
]

In [ ]:
feature_set_C = feature_set_B + [
    'customer_segment_1',
    'customer_segment_2',
    'customer_segment_3'
]

In [ ]:
X_a = X_fs[feature_set_A]
X_b = X_fs[feature_set_B]
X_c = X_fs[feature_set_C]

In [ ]:
y = y_fs
y.value_counts()

In [ ]:
X_c.isnull().sum()

##***6.ML Model Implementation***

###ML models for Feature Set A

In [ ]:
X_a.isnull().sum()

**Train-Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train_A, X_test_A, y_train, y_test = train_test_split(
    X_a,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train_A.shape)
print("X_test shape:", X_test_A.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_test distribution:\n", y_test.value_counts(normalize=True))

Importing libraries for model trainings

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
#Logistic regression pipeline \
#Scaling needed
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ))
])

#Random Forest pipeline
rf_pipeline = Pipeline([
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_estimators=100
    ))
])

#XGBoost Pipeline
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

xgb_pipeline = Pipeline([
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight
    ))
])

In [ ]:
#model dictionary
models_A = {
    'Logistic Regression': logreg_pipeline,
    'Random Forest': rf_pipeline,
    'XGBoost': xgb_pipeline
}

#Training models
for name, model in models_A.items():
    model.fit(X_train_A, y_train)
    print(f"{name} trained successfully.")

####Evaluating the models

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

results_A = []

for name, model in models_A.items():
    # Predictions
    y_pred = model.predict(X_test_A)
    y_proba = model.predict_proba(X_test_A)[:, 1]

    # Store metrics
    results_A.append({
        'Feature_Set': 'A',
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1_Score': round(f1_score(y_test, y_pred), 4),
        'ROC_AUC': round(roc_auc_score(y_test, y_proba), 4)
    })

# Convert to DataFrame
results_A_df = pd.DataFrame(results_A)

# Sort by Recall or F1 if you want
results_A_df.sort_values(by='Recall', ascending=False)

NameError: name 'models_A' is not defined

In [ ]:
for name, model in models_A.items():
    print("=" * 60)
    print(f"Model: {name}")
    print("=" * 60)

    y_pred = model.predict(X_test_A)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("\n")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for name, model in models_A.items():
    y_pred = model.predict(X_test_A)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name} (Feature Set A)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8,6))

for name, model in models_A.items():
    y_proba = model.predict_proba(X_test_A)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Feature Set A")
plt.legend()
plt.show()

###ML Model for Feature Set B

In [ ]:
X_b.isnull().sum()

**Train-Test Split**

In [ ]:
X_train_B, X_test_B, y_train_b, y_test_b = train_test_split(
    X_b,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train_B.shape)
print("X_test shape:", X_test_B.shape)
print("y_train distribution:\n", y_train_b.value_counts(normalize=True))
print("y_test distribution:\n", y_test_b.value_counts(normalize=True))

**Model Training**

In [ ]:
#Logistic regression pipeline \
#Scaling needed
logreg_pipeline_b = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ))
])

#Random Forest pipeline
rf_pipeline_b = Pipeline([
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_estimators=100
    ))
])

#XGBoost Pipeline
scale_pos_weight = y_train_b.value_counts()[0] / y_train_b.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

xgb_pipeline_b = Pipeline([
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight
    ))
])

In [ ]:
#model dictionary
models_B = {
    'Logistic Regression': logreg_pipeline_b,
    'Random Forest': rf_pipeline_b,
    'XGBoost': xgb_pipeline_b
}

#Training models
for name, model in models_B.items():
    model.fit(X_train_B, y_train_b)
    print(f"{name} trained successfully.")

####Evaluating the models

In [ ]:
results_B = []

for name, model in models_B.items():
    # Predictions
    y_pred = model.predict(X_test_B)
    y_proba = model.predict_proba(X_test_B)[:, 1]

    # Store metrics
    results_B.append({
        'Feature_Set': 'B',
        'Model': name,
        'Accuracy': round(accuracy_score(y_test_b, y_pred), 4),
        'Precision': round(precision_score(y_test_b, y_pred), 4),
        'Recall': round(recall_score(y_test_b, y_pred), 4),
        'F1_Score': round(f1_score(y_test_b, y_pred), 4),
        'ROC_AUC': round(roc_auc_score(y_test_b, y_proba), 4)
    })

# Convert to DataFrame
results_B_df = pd.DataFrame(results_B)

# Sort by Recall or F1 if you want
results_B_df.sort_values(by='Recall', ascending=False)

In [ ]:
for name, model in models_B.items():
    y_pred = model.predict(X_test_B)
    cm = confusion_matrix(y_test_b, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name} (Feature Set A)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models_B.items():
    y_proba = model.predict_proba(X_test_B)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_b, y_proba)
    auc_score = roc_auc_score(y_test_b, y_proba)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Feature Set B")
plt.legend()
plt.show()

###ML Model for Feature Set C

In [ ]:
X_c.isnull().sum()

In [ ]:
X_train_C, X_test_C, y_train_c, y_test_c = train_test_split(
    X_c,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train shape:", X_train_C.shape)
print("X_test shape:", X_test_C.shape)
print("y_train distribution:\n", y_train_c.value_counts(normalize=True))
print("y_test distribution:\n", y_test_c.value_counts(normalize=True))

**Model training**

In [ ]:
#Logistic regression pipeline \
#Scaling needed
logreg_pipeline_c = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        random_state=42,
        max_iter=1000
    ))
])

#Random Forest pipeline
rf_pipeline_c = Pipeline([
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_estimators=100
    ))
])

#XGBoost Pipeline
scale_pos_weight = y_train_c.value_counts()[0] / y_train_c.value_counts()[1]
print("scale_pos_weight:", scale_pos_weight)

xgb_pipeline_c = Pipeline([
    ('model', XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight
    ))
])

In [ ]:
models_C = {
    'Logistic Regression': logreg_pipeline_c,
    'Random Forest': rf_pipeline_c,
    'XGBoost': xgb_pipeline_c
}

for name, model in models_C.items():
    model.fit(X_train_C, y_train_c)
    print(f"{name} trained successfully.")

In [ ]:
results_C = []

for name, model in models_C.items():
    # Predictions
    y_pred = model.predict(X_test_C)
    y_proba = model.predict_proba(X_test_C)[:, 1]

    # Store metrics
    results_C.append({
        'Feature_Set': 'C',
        'Model': name,
        'Accuracy': round(accuracy_score(y_test_c, y_pred), 4),
        'Precision': round(precision_score(y_test_c, y_pred), 4),
        'Recall': round(recall_score(y_test_c, y_pred), 4),
        'F1_Score': round(f1_score(y_test_c, y_pred), 4),
        'ROC_AUC': round(roc_auc_score(y_test_c, y_proba), 4)
    })

# Convert to DataFrame
results_C_df = pd.DataFrame(results_C)

# Sort by Recall or F1 if you want
results_C_df.sort_values(by='Recall', ascending=False)

In [ ]:
for name, model in models_C.items():
    y_pred = model.predict(X_test_C)
    cm = confusion_matrix(y_test_c, y_pred)

    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name} (Feature Set A)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))

for name, model in models_C.items():
    y_proba = model.predict_proba(X_test_C)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_c, y_proba)
    auc_score = roc_auc_score(y_test_c, y_proba)

    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison - Feature Set C")
plt.legend()
plt.show()

In [ ]:
final_results_df = pd.concat([results_A_df, results_B_df, results_C_df], ignore_index=True)

final_results_df = final_results_df.sort_values(
    by=['F1_Score', 'Recall', 'ROC_AUC'],
    ascending=False
).reset_index(drop=True)

final_results_df

In [ ]:
best_per_feature_set = (
    final_results_df.sort_values(by='F1_Score', ascending=False)
    .groupby('Feature_Set')
    .head(1)
    .reset_index(drop=True)
)

best_per_feature_set

###**HyperParameter Tuning of Best Performing models**

**These are our best performing models**

XGBoost on Feature Set C

XGBoost on Feature Set B

Random forest on Feature Set B

Random Forest was benchmarked as a tree-based baseline and showed strong precision, but XGBoost consistently outperformed it on recall, F1-score, and ROC-AUC. Therefore, further optimization efforts were focused on XGBoost, which emerged as the stronger churn modeling candidate.

**1.XGBoost on Feature Set B**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_xgb = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [3, 4, 5, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'model__min_child_weight': [1, 3, 5, 7],
    'model__gamma': [0, 0.1, 0.3, 0.5],
    'model__reg_alpha': [0, 0.01, 0.1, 1],
    'model__reg_lambda': [1, 2, 5, 10]
}

random_search_xgb_B = RandomizedSearchCV(
    estimator=xgb_pipeline_b,
    param_distributions=param_dist_xgb,
    n_iter=25,                 # Good balance of quality vs time
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
random_search_xgb_B.fit(X_train_B, y_train_b)

In [ ]:
print("Best Parameters:")
print(random_search_xgb_B.best_params_)

print("\nBest CV F1 Score:")
print(random_search_xgb_B.best_score_)

In [ ]:
best_xgb_B = random_search_xgb_B.best_estimator_

y_pred_xgb_B = best_xgb_B.predict(X_test_B)
y_proba_xgb_B = best_xgb_B.predict_proba(X_test_B)[:, 1]

In [ ]:
print(round(recall_score(y_test_b, y_pred_xgb_B), 4))

In [ ]:
print(classification_report(y_test_b, y_pred_xgb_B))

In [ ]:
cm_xgb_b = confusion_matrix(y_test_b, y_pred_xgb_B)

plt.figure(figsize=(5,4))
sns.heatmap(cm_xgb_b, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - Tuned XGBoost (Feature Set B)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [ ]:
print(roc_auc_score(y_test_b, y_proba_xgb_B))

RandomizedSearchCV was applied to XGBoost on Feature Set B using 5-fold cross-validation and F1-score optimization. The tuned model remained very strong, achieving approximately 0.98 precision, 0.81 recall, and 0.89 F1-score on the churn class, indicating that the baseline model was already close to optimal.

**2.XGBoost on Feature Set C**

In [ ]:
random_search_xgb_C = RandomizedSearchCV(
    estimator=xgb_pipeline_c,
    param_distributions=param_dist_xgb,
    n_iter=25,                 # Good balance of quality vs time
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)
random_search_xgb_C.fit(X_train_C, y_train_c)

In [ ]:
print("Best Parameters:")
print(random_search_xgb_C.best_params_)

print("\nBest CV F1 Score:")
print(random_search_xgb_C.best_score_)

In [ ]:
best_xgb_C = random_search_xgb_C.best_estimator_

y_pred_xgb_C = best_xgb_C.predict(X_test_C)
y_proba_xgb_C = best_xgb_C.predict_proba(X_test_C)[:, 1]

In [ ]:
print(round(recall_score(y_test_c, y_pred_xgb_C), 4))

In [ ]:
print(classification_report(y_test_c, y_pred_xgb_C))

In [ ]:
cm_xgb_c = confusion_matrix(y_test_c, y_pred_xgb_C)

plt.figure(figsize=(5,4))
sns.heatmap(cm_xgb_c, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - Tuned XGBoost (Feature Set C)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [ ]:
print(roc_auc_score(y_test_c, y_proba_xgb_C))

**Although customer segmentation was useful for behavioral profiling and business insight generation, including the segment label as a predictive feature did not improve final model discrimination after tuning. Therefore, I retained segmentation as a strategic business layer while selecting Feature Set B as the final predictive feature space.**

By comparing the metrics like accuracy, precision, recall, F1 score and mainly the roc_auc score, we selected the **Tuned XGBoost model** which was trained on **Feature Set B**

####**Final Comparison of all Models**

In [ ]:
# Tuned XGBoost results
xgb_B_tuned_results = pd.DataFrame([{
    'Feature_Set': 'B',
    'Model': 'XGBoost Tuned',
    'Accuracy': 0.9718,      # replace if needed
    'Precision': 0.9898,
    'Recall': 0.8083,
    'F1_Score': 0.8899,
    'ROC_AUC': 0.9150
}])

xgb_C_tuned_results = pd.DataFrame([{
    'Feature_Set': 'C',
    'Model': 'XGBoost Tuned',
    'Accuracy': 0.9729,      # replace if needed
    'Precision': 1.0000,
    'Recall': 0.8083,
    'F1_Score': 0.8940,
    'ROC_AUC': 0.9004
}])

# Filter final serious baseline contenders
final_baseline_models = final_results_df[
    (
        ((final_results_df['Feature_Set'] == 'B') &
         (final_results_df['Model'].isin(['Logistic Regression', 'Random Forest', 'XGBoost'])))
        |
        ((final_results_df['Feature_Set'] == 'C') &
         (final_results_df['Model'].isin(['XGBoost'])))
    )
].copy()

# Combine everything
final_comparison_df = pd.concat(
    [final_baseline_models, xgb_B_tuned_results, xgb_C_tuned_results],
    ignore_index=True
)

# Sort by strongest balance
final_comparison_df = final_comparison_df.sort_values(
    by=['F1_Score', 'ROC_AUC'],
    ascending=False
).reset_index(drop=True)

# Display
display(final_comparison_df)

# Optional pretty display
final_comparison_df.style.background_gradient(
    cmap='Blues',
    subset=['Accuracy', 'Precision', 'Recall', 'F1_Score', 'ROC_AUC']
)

Feature Set C showed marginally better threshold-dependent metrics like F1-score, but the improvement was very small and came from adding a derived clustering-based segment feature. After tuning, Feature Set B delivered the strongest ROC-AUC, which was important because churn prediction is often used to rank customers by risk rather than only classify them at a fixed threshold. Since both feature sets achieved the same recall, I selected Tuned XGBoost on Feature Set B as the final model because it offered better discrimination performance while keeping the predictive feature space cleaner and more directly explainable.

In [ ]:
# Predicted probabilities for tuned models
xgb_b_tuned_proba = best_xgb_B.predict_proba(X_test_B)[:, 1]
xgb_c_tuned_proba = best_xgb_C.predict_proba(X_test_C)[:, 1]

# ROC curve values
fpr_b_tuned, tpr_b_tuned, _ = roc_curve(y_test_b, xgb_b_tuned_proba)
fpr_c_tuned, tpr_c_tuned, _ = roc_curve(y_test_c, xgb_c_tuned_proba)

# AUC scores
auc_b_tuned = roc_auc_score(y_test_b, xgb_b_tuned_proba)
auc_c_tuned = roc_auc_score(y_test_c, xgb_c_tuned_proba)

# Plot
plt.figure(figsize=(8, 6))

plt.plot(fpr_b_tuned, tpr_b_tuned, label=f'Tuned XGBoost - Feature Set B (AUC = {auc_b_tuned:.4f})')
plt.plot(fpr_c_tuned, tpr_c_tuned, label=f'Tuned XGBoost - Feature Set C (AUC = {auc_c_tuned:.4f})')

# Random classifier line
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')

plt.title('ROC Curve Comparison: Tuned Final Contenders')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

To compare the final tuned candidates, ROC curves were plotted for Tuned XGBoost models on Feature Set B and Feature Set C. Although both models showed strong classification performance, Tuned XGBoost on Feature Set B achieved the higher ROC-AUC score, indicating better overall ability to rank customers by churn risk across thresholds. This supported its selection as the final predictive model.

In [ ]:
# Extract feature importance from tuned XGBoost model
feature_importance_df = pd.DataFrame({
    'Feature': X_train_B.columns,
    'Importance': best_xgb_B.named_steps['model'].feature_importances_
})

# Sort descending
feature_importance_df = feature_importance_df.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

feature_importance_df.head(15)

In [ ]:
plt.figure(figsize=(10, 7))

sns.barplot(
    data=feature_importance_df.head(15),
    x='Importance',
    y='Feature'
)

plt.title('Top 15 Feature Importances - Final Tuned XGBoost Model (Feature Set B)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()